# Lyapunov function validation

## Definition

A [Lyapunov function](https://en.wikipedia.org/wiki/Lyapunov_function) for an autonomous dynamical system described by

$\begin{cases} 
g: \mathbb{R}^n \rightarrow \mathbb{R}^n, \\
\dot y = g(y) \end{cases}$

with equalibrium point at $y = 0$ is a scalar function $V:\mathbb{R}^n \rightarrow \mathbb{R}$ that is:
* Continuous
* Has continuous first derivatives
* Is strictly positive for $y \neq 0$
* Time derivative $\dot V = \Delta V \cdot g$ is non-positive.

## Required packages

In [14]:
import cvxpy as cp
import numpy as np
import cvxopt # needs installed for optimization

## Polynomial example

Consider the dynamical system described by $\dot x = -x$. 

Use SDP to verify the 4th condition for $V(x) = x^2$ is a valid Lyapunov function for this system.

(Hint 1: Consider the form $f(t) = p_i t^i$)

(Hint 2: [Theorem 11.2](https://www.damtp.cam.ac.uk/user/hf323/lent2017_topics_convex_optimisation/lecture11.pdf))

In [15]:
# Define the vector P
# (take negative, f(t) >= 0 iff -f(t) <= 0)
P = (0, 0, 2)
k = 1

# Add the PSD variable
X = cp.Variable((k+1,k+1), PSD=True)

# Zero objective for primal problem
zero_matrix = np.zeros((k+1,k+1))
objective = cp.trace(zero_matrix @ X)

# Set up the constraints
constraints = []
for d in range(2*k + 1):
    lhs = cp.sum([X[i,j] for i in range(k+1) for j in range(k+1) if i+j == d])
    constraints += [lhs == P[d]]

# Set and solve problem
problem = cp.Problem(
    cp.Maximize(objective),
    constraints
)
problem.solve(solver = 'CVXOPT')
print("Status:", problem.status)

Status: optimal


## Function for checking

If you have not already, construct a function that takes a tuple, $P$, representing a polynomial and checks if it is non-negative.

(Hint 3: Name the function `check_poly_nonnegative(P)` and ensure it returns boolean values, `True` if non-negative and `False` if not)

In [ ]:
def check_poly_nonnegative(P: tuple) -> bool:
    # Ensuring last digit is non-zero
    while P[-1] == 0:
        P = P[:-1]
        if len(P) == 0:
            return True # Zero poly
        
    # Checking its not a negative constant
    if len(P) == 1:
        if P[0] >= 0:
            return True
        else:
            return False

    # Ensuring it is an even degree
    if len(P) % 2 == 0: 
        return False
    
    # Getting the value for k
    k = int((len(P)-1)/2)

    # Add the PSD variable
    X = cp.Variable((k+1,k+1), PSD=True)

    # Zero objective for primal problem
    zero_matrix = np.zeros((k+1,k+1))
    objective = cp.trace(zero_matrix @ X)

    # Set up the constraints
    constraints = []
    for d in range(2*k + 1):
        lhs = cp.sum([X[i,j] for i in range(k+1) for j in range(k+1) if i+j == d])
        constraints += [lhs == P[d]]

    # Set and solve problem
    problem = cp.Problem(
        cp.Maximize(objective),
        constraints
    )
    problem.solve(solver = 'CVXOPT')
    
    # Return outcome
    if problem.status == "optimal":
        return True
    elif problem.status == "infeasible":
        return False
    else:
        raise Exception(f"Problem has unexpected status: {problem.status}")

## Polynomials to check

The following $P$ vectors represent different polynomials in different sized spaces. Check if your function checks them correctly.

In [16]:
polynomials = {
    (0,0,1): True, # x^2
    (0,0,-1): False, # -x^2
    (0,0,0): True, # 0
    (0,1): False, # x
    (1,1,1): True, # 1 + x + x^2
    (0,1,0): False, # x
    (0,1,1,0): False, # x + x^2
    (0,1,-1,0): False, # x + -x^2
}

for i, P in enumerate(polynomials):
    print(f"Polynomial check {i+1} passed: {check_poly_nonnegative(P) == polynomials[P]}")

Polynomial check 1 passed: True
Polynomial check 2 passed: True
Polynomial check 3 passed: True
Polynomial check 4 passed: True
Polynomial check 5 passed: True
Polynomial check 6 passed: True
Polynomial check 7 passed: True
Polynomial check 8 passed: True


## Application in ???

Need to find a polynomial lyapunov function for something